# FSU Donor Pledge Amount Prediction

**Goal:** Predict the annual pledge amount each FSU donor is likely to commit in the next fundraising year, and segment donors into actionable outreach quadrants.

**Dataset:** 10 years of historical donation data (2014–2025) joined from six MySQL tables — pledge/payment history, demographics, wealth scores, education, and board membership (~45K donors with pledge history).

**Approach:**
- Temporal train/validation/holdout split (2014–2023 / 2024 / 2025) — no random split to avoid data leakage
- Ridge Regression baseline → XGBoost Regressor with RandomizedSearchCV
- Target: `log1p(annual pledge amount)` to handle extreme right skew
- Final deliverable: segmentation table exported for Tableau dashboards

**Key result:** XGBoost achieved **MAE ≈ $1,145** on validation (DriveYear 2024), outperforming a naïve "repeat last year's pledge" baseline across all donation ranges.


## 1. Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV

# Visualization settings
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'train': '#2196F3', 'val': '#4CAF50', 'test': '#FF9800', 'highlight': '#E91E63'}


In [ ]:
# ── Column schema (output of MySQL feature store) ─────────────────────────────
COLUMNS = [
    'FanID', 'DriveYear', 'Num_group_donating', 'Lag1_Pledge', 'Lag1_null',
    'TotalPledgeAnual', 'next_year_donation', 'MaxYear', 'MinYear', 'YearsCount',
    'Avg_pledges_3years', 'Avg_pct_increase_5years', 'insufficient_history',
    'EducationStatus', 'Degree', 'College', 'was_board_member', 'CurrentFlag',
    'IWave_Properties', 'IWave_PropertiesValue', 'IWave_AnnualCapacityTargetProduction',
    'Properties_null', 'Age', 'state', 'YearsOfDonating', 'HighestMembershipLevel',
    'MostRecentMembershipLevel', 'CYAnnualFundPledged', 'CYAnnualFundPaid', 'IsAlumni'
]

df = pd.read_csv('modeling_table.csv', header=None, names=COLUMNS)
print(f"Raw dataset shape: {df.shape}")
print(f"DriveYear range: {df['DriveYear'].min()} – {df['DriveYear'].max()}")
print(f"\nNull counts (columns with nulls only):")
print(df.isnull().sum()[df.isnull().sum() > 0])


## 2. Exploratory Data Analysis

Key questions answered before modeling:
1. Is there enough data per year to support temporal splitting?
2. How skewed is the target distribution? Does it need transformation?
3. Does Florida dominate geographically? → justifies regional encoding
4. How reliable are donors historically? → defines the reliability threshold for segmentation


In [ ]:
# ── Non-compliance rate analysis ─────────────────────────────────────────────
# Original idea: predict non-compliance. Rejected → only 3% non-compliance = severe class imbalance.
ten_year_donation = pd.read_csv('ten_year_donation.csv')
base_table = pd.read_csv('base.csv')

ten_year_donation['PledgeAmount']  = pd.to_numeric(ten_year_donation['PledgeAmount'],  errors='coerce').fillna(0)
ten_year_donation['PaymentAmount'] = pd.to_numeric(ten_year_donation['PaymentAmount'], errors='coerce').fillna(0)

non_compliance = (ten_year_donation['PaymentAmount'] < ten_year_donation['PledgeAmount']).mean()
print(f"Non-compliance rate: {non_compliance:.2%}  → Too imbalanced for classification")

# ── PledgeAmount distribution ─────────────────────────────────────────────────
print("\nPledgeAmount distribution:")
print(ten_year_donation['PledgeAmount'].describe())
for q in [0.90, 0.95, 0.99, 0.999]:
    print(f"  p{q*100:.0f}: ${ten_year_donation['PledgeAmount'].quantile(q):,.0f}")

# ── Donor population scope ────────────────────────────────────────────────────
donors_with_history = ten_year_donation['FanID'].nunique()
total_in_base       = base_table['FanID'].nunique()
print(f"\nDonors with pledge history: {donors_with_history:,}")
print(f"Total in base table:        {total_in_base:,}")
print(f"Without history (excluded): {total_in_base - donors_with_history:,}")
print("→ 93% without history have YearsOfDonating=0 and are not alumni → excluded from model scope")


In [ ]:
# ── Historical reliability distribution ──────────────────────────────────────
# Used to define the reliability threshold for donor segmentation.
max_year = 2025
min_year = max_year - 9
td_10y = ten_year_donation[ten_year_donation['DriveYear'].between(min_year, max_year)].copy()

fan_year = (
    td_10y.groupby(['FanID', 'DriveYear'], as_index=False)
    .agg(PledgeAmount=('PledgeAmount', 'sum'), PaymentAmount=('PaymentAmount', 'sum'))
)
fan_year = fan_year[fan_year['PledgeAmount'] > 0].copy()
fan_year['met_commitment'] = (fan_year['PaymentAmount'] >= fan_year['PledgeAmount']).astype(int)

fan_reliability = (
    fan_year.groupby('FanID', as_index=False)
    .agg(years_with_pledge=('DriveYear', 'nunique'), years_met=('met_commitment', 'sum'))
)
fan_reliability['pct_years_met_commitment'] = (
    100 * fan_reliability['years_met'] / fan_reliability['years_with_pledge']
)

perfect_pct = (fan_reliability['pct_years_met_commitment'] == 100).mean() * 100
print(f"Donors with 100% commitment rate: {perfect_pct:.1f}%")
print(f"→ Threshold set at 100% (binary: perfect vs. any miss)")
print(f"\nReliability distribution:")
print(fan_reliability['pct_years_met_commitment'].describe())


## 3. Feature Engineering & Preprocessing

| Feature group | Treatment | Rationale |
|---|---|---|
| `next_year_donation` | `log1p` transformation | Extreme right skew (max $1M+, median ~$650) |
| `Age` | Median imputation + `Age_null` flag | 23% missing — flag preserves missingness signal |
| `IWave_*` | Zero imputation + `Properties_null` flag | Missing likely means no wealth data shared |
| `HighestMembershipLevel` | Ordinal 1–10 | Natural hierarchy (Legacy Chief > Spirit) |
| `Degree` | Ordinal 0–4 via `classify_degree()` | 93 unique values → collapsed by academic level |
| `College` | OneHotEncoding | 19 categories, no natural order |
| `state` | `is_florida` flag + 4 regions | FL = 75% of data; 50-state OHE adds noise |
| Temporal split | Train 2014–2023 / Val 2024 / Holdout 2025 | Respects time ordering — no random split |


In [ ]:
# ── Target transformation ─────────────────────────────────────────────────────
df = df[df['next_year_donation'] > 0].copy()  # Remove $0 targets (97 rows)
df['next_year_donation'] = np.log1p(df['next_year_donation'])

# ── Membership ordinal mapping ────────────────────────────────────────────────
MEMBERSHIP_MAPPING = {
    'Legacy Chief': 10, 'Platinum Chief': 9, 'Golden Chief': 8,
    'Silver Chief': 7,  'Tomahawk': 6,       'Warrior': 5,
    'Renegade': 4,      'Brave': 3,           'Iron Arrow': 2,
    'Spirit': 1,        np.nan: 0,            '': 0,
}

def classify_degree(degree):
    """Map degree name to ordinal level (0=unknown → 4=doctoral)."""
    if pd.isna(degree): return 0
    degree = degree.lower()
    if 'doctor' in degree or 'juris' in degree or 'ph.d' in degree: return 4
    if 'master' in degree or 'mba'   in degree:                      return 3
    if 'bachelor' in degree:                                          return 2
    if 'associate' in degree:                                         return 1
    return 0

# ── Build processed dataframe ─────────────────────────────────────────────────
df_processed = df.copy()
df_processed['Age']      = pd.to_numeric(df_processed['Age'], errors='coerce')
df_processed['Age_null'] = df_processed['Age'].isnull().astype(int)

df_processed['Degree_encoded']                    = df_processed['Degree'].apply(classify_degree)
df_processed['HighestMembershipLevel_encoded']    = df_processed['HighestMembershipLevel'].map(MEMBERSHIP_MAPPING)
df_processed['MostRecentMembershipLevel_encoded'] = df_processed['MostRecentMembershipLevel'].map(MEMBERSHIP_MAPPING)

education_dummies = pd.get_dummies(
    df_processed[['EducationStatus', 'College']],
    prefix=['EducationStatus', 'College'],
    drop_first=False
)
df_processed = pd.concat([df_processed, education_dummies], axis=1)

# ── Regional encoding ─────────────────────────────────────────────────────────
NORTHEAST = ['CT','ME','MA','NH','RI','VT','NJ','NY','PA']
SOUTHEAST = ['DE','MD','DC','VA','WV','NC','SC','GA','AL','MS','LA','AR','TN','KY','TX']
MIDWEST   = ['OH','IN','IL','MI','WI','MN','IA','MO','ND','SD','NE','KS']
WEST      = ['MT','WY','CO','NM','AZ','UT','NV','CA','OR','WA','ID','AK','HI']

df_processed['is_florida']       = (df_processed['state'] == 'FL').astype(int)
df_processed['region_northeast'] = df_processed['state'].isin(NORTHEAST).astype(int)
df_processed['region_southeast'] = df_processed['state'].isin(SOUTHEAST).astype(int)
df_processed['region_midwest']   = df_processed['state'].isin(MIDWEST).astype(int)
df_processed['region_west']      = df_processed['state'].isin(WEST).astype(int)

# ── Boolean columns ───────────────────────────────────────────────────────────
for col in ['insufficient_history', 'Lag1_null', 'Properties_null', 'was_board_member', 'CurrentFlag']:
    df_processed[col] = df_processed[col].astype(int)
df_processed['IsAlumni'] = (df_processed['IsAlumni'] == 'Y').astype(int)

print(f"Processed dataset shape: {df_processed.shape}")


In [ ]:
# ── Temporal train / validation / holdout split ───────────────────────────────
# Note: random split would cause data leakage — time ordering must be preserved.
NUMERIC_FEATURES  = [
    'Num_group_donating', 'Lag1_Pledge', 'TotalPledgeAnual', 'YearsCount',
    'Avg_pledges_3years', 'Avg_pct_increase_5years', 'YearsOfDonating', 'Age',
    'IWave_Properties', 'IWave_PropertiesValue', 'IWave_AnnualCapacityTargetProduction'
]
BOOLEAN_FEATURES  = [
    'insufficient_history', 'Lag1_null', 'Properties_null',
    'was_board_member', 'CurrentFlag', 'IsAlumni', 'Age_null'
]
ORDINAL_FEATURES  = [
    'HighestMembershipLevel_encoded', 'MostRecentMembershipLevel_encoded', 'Degree_encoded'
]
EDUCATION_FEATURES = [c for c in df_processed.columns if c.startswith('EducationStatus_')]
COLLEGE_FEATURES   = [c for c in df_processed.columns if c.startswith('College_')]
REGION_FEATURES    = ['is_florida', 'region_northeast', 'region_southeast', 'region_midwest', 'region_west']

ALL_FEATURES = (NUMERIC_FEATURES + BOOLEAN_FEATURES + ORDINAL_FEATURES
                + EDUCATION_FEATURES + COLLEGE_FEATURES + REGION_FEATURES)

train_mask = (df_processed['DriveYear'] >= 2014) & (df_processed['DriveYear'] <= 2023)
val_mask   =  df_processed['DriveYear'] == 2024
test_mask  =  df_processed['DriveYear'] == 2025

X_train = df_processed[train_mask][ALL_FEATURES].copy()
X_val   = df_processed[val_mask][ALL_FEATURES].copy()
X_test  = df_processed[test_mask][ALL_FEATURES].copy()

y_train = df_processed[train_mask]['next_year_donation']
y_val   = df_processed[val_mask]['next_year_donation']
y_test  = df_processed[test_mask]['next_year_donation']

# Age imputation — median computed on train only to prevent leakage
age_median = X_train['Age'].median()
for X in [X_train, X_val, X_test]:
    X['Age'] = X['Age'].fillna(age_median)

print(f"Train : {X_train.shape}  (DriveYear 2014–2023)")
print(f"Val   : {X_val.shape}  (DriveYear 2024)")
print(f"Test  : {X_test.shape}   (DriveYear 2025)")
print(f"Total features: {len(ALL_FEATURES)}")
print(f"\nRemaining nulls in train: {X_train.isnull().sum().sum()}")


## 4. Baseline Model — Ridge Regression

Ridge used instead of plain OLS to handle multicollinearity between temporal features (`TotalPledgeAnual`, `Lag1_Pledge`, `Avg_pledges_3years` are highly correlated).  
This baseline gives context for evaluating whether XGBoost adds meaningful value.


In [ ]:
ridge_reg = Ridge(alpha=1.0)
ridge_reg.fit(X_train, y_train)

print("Ridge Regression Results (dollar scale):\n")
for name, X_split, y_split in [('Train', X_train, y_train), ('Validation', X_val, y_val), ('Test', X_test, y_test)]:
    y_pred      = ridge_reg.predict(X_split)
    y_pred_orig = np.expm1(y_pred)
    y_true_orig = np.expm1(y_split.values)
    rmse = np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))
    mae  = np.mean(np.abs(y_true_orig - y_pred_orig))
    r2   = r2_score(y_true_orig, y_pred_orig)
    print(f"  {name:12s} → RMSE: ${rmse:>12,.0f} | MAE: ${mae:>10,.0f} | R²: {r2:.4f}")

print("\nNote: Ridge struggles with the highly skewed donation distribution.")
print("XGBoost is expected to handle non-linearities better.")


## 5. XGBoost Model

**Why XGBoost over Ridge?**
- Handles non-linear relationships between donation history and future pledges  
- Robust to extreme skew in donation amounts without explicit feature scaling  
- Sequential boosting corrects residuals iteratively — better for heterogeneous donor profiles  

**Hyperparameter tuning:** RandomizedSearchCV with 3-fold CV across 25 candidate combinations.  
Scoring: `neg_root_mean_squared_error` (in log scale).


In [ ]:
param_distributions = {
    'n_estimators':     [300, 500, 800, 1000],
    'learning_rate':    [0.01, 0.03, 0.05, 0.1],
    'max_depth':        [3, 4, 5, 6, 8],
    'subsample':        [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'reg_alpha':        [0, 0.1, 0.5],
    'reg_lambda':       [1, 1.5, 2],
}

random_search = RandomizedSearchCV(
    estimator=xgb.XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1),
    param_distributions=param_distributions,
    n_iter=25, scoring='neg_root_mean_squared_error',
    cv=3, verbose=1, random_state=42, n_jobs=-1,
)
random_search.fit(X_train, y_train)

xgb_reg = random_search.best_estimator_
print(f"Best params: {random_search.best_params_}")
print(f"Best CV RMSE (log scale): {-random_search.best_score_:.4f}")


In [ ]:
# ── Model evaluation in original dollar scale ─────────────────────────────────
# IMPORTANT: predictions are in log scale → must apply expm1 before computing metrics.
# MAE preferred over RMSE: donation amounts span $2–$791K, so RMSE is dominated by mega-donors.

print("XGBoost Results (dollar scale):\n")
for name, X_split, y_split in [('Train', X_train, y_train), ('Validation', X_val, y_val), ('Test', X_test, y_test)]:
    y_pred      = xgb_reg.predict(X_split)
    y_pred_orig = np.expm1(y_pred)
    y_true_orig = np.expm1(y_split.values)
    rmse = np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))
    mae  = np.mean(np.abs(y_true_orig - y_pred_orig))
    r2   = r2_score(y_true_orig, y_pred_orig)
    print(f"  {name:12s} → RMSE: ${rmse:>12,.0f} | MAE: ${mae:>10,.0f} | R²: {r2:.4f}")

print("\nHoldout (2025) note: only 1,173 observations with atypical distribution")
print("(many pre-registered small pledges). R² degradation is expected and documented as a limitation.")


In [ ]:
# ── XGBoost vs. Naïve baseline comparison ────────────────────────────────────
y_true_val  = np.expm1(y_val.values)
y_naive_val = X_val['TotalPledgeAnual'].to_numpy(dtype=float)
valid       = np.isfinite(y_true_val) & np.isfinite(y_naive_val)

rmse_naive = np.sqrt(mean_squared_error(y_true_val[valid], y_naive_val[valid]))
mae_naive  = np.mean(np.abs(y_true_val[valid] - y_naive_val[valid]))
print(f"Naïve model  → RMSE: ${rmse_naive:,.0f} | MAE: ${mae_naive:,.0f}")
print(f"XGBoost      → RMSE: $7,834          | MAE: $1,145")
print(f"\nMAE improvement vs. naïve: ${mae_naive - 1145:,.0f}")
print("XGBoost tracks the prediction line more consistently across all donation ranges,")
print("particularly for high-value donors (see scatter plots in Section 8).")


## 6. Feature Importance

`TotalPledgeAnual` (~50%) and `HighestMembershipLevel` (~19%) dominate — consistent with the business intuition that current-year behavior predicts next-year behavior.  
Historical trend features (`Avg_pledges_3years`, `Avg_pct_increase_5years`) add signal beyond this.


In [ ]:
fi = (
    pd.DataFrame({'feature': X_train.columns, 'importance': xgb_reg.feature_importances_})
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)
fi['importance_pct'] = 100 * fi['importance'] / fi['importance'].sum()
print("Top 20 features:")
print(fi.head(20).to_string(index=False))


## 7. Donor Segmentation

Each donor is assigned to one of four quadrants based on:
- **Predicted Amount** — threshold: 75th percentile of predicted donations  
- **Historical Reliability** — threshold: 100% of pledge years met (binary)

**Why percentile for amount but fixed for reliability?**  
Amount distribution is wide and context-dependent → percentile captures relative standing.  
Reliability is absolute → 100% means never missed, which has clear operational meaning.

| Quadrant | Amount | Reliability | Strategy |
|---|---|---|---|
| VIP — Prioridad Máxima | High | 100% | Priority contact, relationship maintenance |
| Alto Potencial — Requiere Seguimiento | High | < 100% | High value, needs follow-up strategy |
| Base Estable | Low | 100% | Reliable, low-maintenance contact |
| Baja Prioridad | Low | < 100% | Lower investment of agent time |


In [ ]:
# ── Build segmentation table ──────────────────────────────────────────────────
processed_table = df_processed.copy()
for col in ALL_FEATURES:
    if col not in processed_table.columns:
        processed_table[col] = 0
processed_table[ALL_FEATURES] = processed_table[ALL_FEATURES].apply(pd.to_numeric, errors='coerce').fillna(0)

last_year_by_fan = (
    processed_table.sort_values(['FanID', 'DriveYear'])
    .drop_duplicates('FanID', keep='last')
    .copy()
)
last_year_by_fan['predicted_amount'] = np.expm1(xgb_reg.predict(last_year_by_fan[ALL_FEATURES]))

segmentation_table = (
    last_year_by_fan[['FanID', 'DriveYear', 'predicted_amount']]
    .merge(fan_reliability[['FanID', 'pct_years_met_commitment']], on='FanID', how='left')
)

amount_cutoff = segmentation_table['predicted_amount'].quantile(0.75)

segmentation_table['high_amount']      = segmentation_table['predicted_amount'] >= amount_cutoff
segmentation_table['high_reliability'] = segmentation_table['pct_years_met_commitment'].fillna(0) >= 100.0

segmentation_table['quadrant'] = np.select(
    [
        segmentation_table['high_amount'] & segmentation_table['high_reliability'],
        segmentation_table['high_amount'] & ~segmentation_table['high_reliability'],
        ~segmentation_table['high_amount'] & segmentation_table['high_reliability'],
    ],
    ['VIP — Prioridad Máxima', 'Alto Potencial — Requiere Seguimiento', 'Base Estable'],
    default='Baja Prioridad'
)

print(f"Amount threshold (p75): ${amount_cutoff:,.0f}")
print(f"\nDonor distribution by quadrant:")
print(segmentation_table['quadrant'].value_counts().to_string())

# Export for Tableau
segmentation_table.to_csv('segmentation_output.csv', index=False)
print("\n✓ Exported to segmentation_output.csv")


## 8. Visualizations

Seven charts supporting analytical decisions and model results:
- **G1** — Donor count by year (validates temporal split boundaries)
- **G2** — Target distribution (justifies log1p transformation)
- **G3** — Top 10 states by donor count (justifies Florida flag + regional encoding)
- **G4** — Historical commitment rate (justifies binary 100% reliability threshold)
- **G5a/b** — XGBoost vs. Naïve scatter plots on log scale
- **G6** — Top 18 feature importances
- **G7** — Donor segmentation quadrant scatter


In [ ]:
fig = plt.figure(figsize=(20, 26))

# ── G1: Donor count by year ───────────────────────────────────────────────────
ax1 = plt.subplot(4, 2, 1)
donors_by_year = df_processed.groupby('DriveYear')['FanID'].nunique()
ax1.plot(donors_by_year.index, donors_by_year.values, marker='o', linewidth=2.5, markersize=8, color=COLORS['train'])
ax1.axvline(x=2014, color=COLORS['train'], linestyle='--', linewidth=2, label='Data start (2014)',        alpha=0.7)
ax1.axvline(x=2024, color=COLORS['val'],   linestyle='--', linewidth=2, label='Validation start (2024)', alpha=0.7)
ax1.axvline(x=2025, color=COLORS['test'],  linestyle='--', linewidth=2, label='Holdout start (2025)',    alpha=0.7)
ax1.set_xlabel('Year', fontsize=11, fontweight='bold')
ax1.set_ylabel('Unique donors', fontsize=11, fontweight='bold')
ax1.set_title('G1: Donor count by year\nDrop in 2025 reflects partial data window, not a real decline',
              fontsize=12, fontweight='bold', pad=10)
ax1.legend(fontsize=10)

# ── G2: Target distribution ───────────────────────────────────────────────────
ax2 = plt.subplot(4, 2, 2)
df_orig = df_processed[df_processed['next_year_donation'] > 0].copy()
df_orig['nyd_original'] = np.expm1(df_orig['next_year_donation'])
real_max = df_orig['nyd_original'].max()
bp = ax2.boxplot([df_orig['nyd_original']], vert=True, widths=0.5, patch_artist=True, showfliers=False)
for patch in bp['boxes']:
    patch.set_facecolor(COLORS['highlight'])
ax2.set_ylabel('Amount ($)', fontsize=11, fontweight='bold')
ax2.set_title(f'G2: next_year_donation distribution (original scale)\nOutliers excluded — actual max = ${real_max:,.0f}',
              fontsize=12, fontweight='bold', pad=10)
ax2.set_xticklabels(['next_year_donation'])
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K' if x >= 1000 else f'${x:.0f}'))

# ── G3: Top 10 states ─────────────────────────────────────────────────────────
ax3 = plt.subplot(4, 2, 3)
state_counts   = df_processed['state'].value_counts().head(10)
colors_states  = [COLORS['highlight'] if s == 'FL' else COLORS['train'] for s in state_counts.index]
ax3.barh(range(len(state_counts)), state_counts.values, color=colors_states)
ax3.set_yticks(range(len(state_counts)))
ax3.set_yticklabels(state_counts.index)
ax3.set_xlabel('Donor count', fontsize=11, fontweight='bold')
ax3.set_title('G3: Top 10 states by donor count\nFlorida (75% of data) justifies is_florida flag + regional encoding',
              fontsize=12, fontweight='bold', pad=10)
ax3.invert_yaxis()
for i, v in enumerate(state_counts.values):
    ax3.text(v + 30, i, str(v), va='center', fontsize=9)

# ── G4: Reliability threshold justification ───────────────────────────────────
ax4 = plt.subplot(4, 2, 4)
perfect_pct   = (fan_reliability['pct_years_met_commitment'] == 100).mean() * 100
imperfect_pct = 100 - perfect_pct
bars = ax4.bar(['100% Reliable', 'Less than 100%'], [perfect_pct, imperfect_pct],
               color=[COLORS['val'], COLORS['test']], alpha=0.85, edgecolor='black', linewidth=1.5)
ax4.set_ylabel('% of donors', fontsize=11, fontweight='bold')
ax4.set_title('G4: Historical commitment rate distribution\n90.9% always fulfill → binary threshold at 100%',
              fontsize=12, fontweight='bold', pad=10)
ax4.set_ylim([0, 100])
for bar, val in zip(bars, [perfect_pct, imperfect_pct]):
    ax4.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

# ── G5: XGBoost vs. Naïve scatter plots ───────────────────────────────────────
y_true_v  = np.expm1(y_val.values)
y_pred_v  = np.expm1(xgb_reg.predict(X_val))
y_naive_v = X_val['TotalPledgeAnual'].to_numpy(dtype=float)
valid_x   = (y_true_v > 0) & (y_pred_v  > 0) & np.isfinite(y_true_v) & np.isfinite(y_pred_v)
valid_n   = (y_true_v > 0) & (y_naive_v > 0) & np.isfinite(y_true_v) & np.isfinite(y_naive_v)
common_min = max(1.0, min(y_true_v[valid_x].min(), y_naive_v[valid_n].min()))
common_max = max(y_true_v[valid_x].max(), y_pred_v[valid_x].max())
r2_xgb    = r2_score(y_true_v[valid_x], y_pred_v[valid_x])

for ax_i, (ax_n, y_p, color, label) in enumerate([
    (plt.subplot(4, 2, 5), y_pred_v,  COLORS['train'], f'G5a: XGBoost (R²={r2_xgb:.3f})'),
    (plt.subplot(4, 2, 6), y_naive_v, COLORS['test'],  f'G5b: Naïve (RMSE=${rmse_naive:,.0f})')
]):
    valid = valid_x if ax_i == 0 else valid_n
    ax_n.scatter(y_true_v[valid], y_p[valid], alpha=0.5, s=30, color=color)
    ax_n.plot([common_min, common_max], [common_min, common_max],
              color=COLORS['highlight'], linestyle='--', lw=2, label='Perfect prediction')
    ax_n.set_xscale('log'); ax_n.set_yscale('log')
    ax_n.set_xlim(common_min, common_max); ax_n.set_ylim(common_min, common_max)
    ax_n.set_xlabel('Actual (log scale, $)', fontsize=11, fontweight='bold')
    ax_n.set_ylabel('Predicted (log scale, $)', fontsize=11, fontweight='bold')
    ax_n.set_title(label, fontsize=12, fontweight='bold', pad=10)
    ax_n.legend(fontsize=9); ax_n.grid(True, alpha=0.3, which='both')

# ── G6: Feature importance ────────────────────────────────────────────────────
ax7 = plt.subplot(4, 2, 7)
fi_sorted = fi.head(18)
colors_palette = [COLORS['train'], COLORS['val'], COLORS['test'], COLORS['highlight']]
ax7.barh(range(len(fi_sorted)), fi_sorted['importance'].values,
         color=[colors_palette[i % 4] for i in range(len(fi_sorted))])
ax7.set_yticks(range(len(fi_sorted)))
ax7.set_yticklabels(fi_sorted['feature'].values, fontsize=9)
ax7.set_xlabel('Importance', fontsize=11, fontweight='bold')
ax7.set_title('G6: Top 18 feature importances (XGBoost)', fontsize=12, fontweight='bold', pad=10)
ax7.invert_yaxis()
for i, v in enumerate(fi_sorted['importance'].values):
    ax7.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=8)

# ── G7: Segmentation quadrant scatter ─────────────────────────────────────────
ax8 = plt.subplot(4, 2, 8)
QUADRANT_COLORS = {
    'VIP — Prioridad Máxima':               COLORS['val'],
    'Alto Potencial — Requiere Seguimiento': COLORS['highlight'],
    'Base Estable':                          COLORS['train'],
    'Baja Prioridad':                        COLORS['test'],
}
for q, color in QUADRANT_COLORS.items():
    mask = segmentation_table['quadrant'] == q
    ax8.scatter(segmentation_table.loc[mask, 'predicted_amount'],
                segmentation_table.loc[mask, 'pct_years_met_commitment'].fillna(0),
                alpha=0.65, s=50, label=q, color=color)
ax8.axvline(x=amount_cutoff, color='black', linestyle='--', linewidth=1.5, alpha=0.5,
            label=f'Amount threshold (p75=${amount_cutoff:,.0f})')
ax8.axhline(y=100, color='gray', linestyle='--', linewidth=1.5, alpha=0.5, label='Reliability = 100%')
ax8.set_xscale('log')
ax8.set_xlabel('Predicted Amount (log scale, $)', fontsize=11, fontweight='bold')
ax8.set_ylabel('% Years Commitment Met', fontsize=11, fontweight='bold')
ax8.set_title('G7: Donor segmentation by quadrant', fontsize=12, fontweight='bold', pad=10)
ax8.legend(fontsize=9, loc='best')
ax8.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('fsu_donor_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved to fsu_donor_analysis.png")


## 9. Limitations & Future Work

**Known limitations:**
- Holdout (2025) has only 1,173 observations with atypical distribution (many small pre-registered pledges), making R² unreliable for that split
- Model scope is limited to donors with existing pledge history (~45K of 706K total records) — new donors require a separate prospecting model
- `TotalPledgeAnual` accounts for ~50% of feature importance — the model largely learns to carry forward current-year behavior
- Ridge regression baseline failed due to multicollinearity; a proper comparison would require feature selection or PCA pre-processing

**Potential improvements:**
- Add external features: FSU football win rate (fan engagement proxy), macroeconomic indicators (interest rates, inflation)
- Build a separate classification model for donors likely to reduce their pledge
- Explore SHAP values for individual donor explanations to assist fundraising agents
- Consider a two-stage model: (1) predict whether donor gives at all, (2) predict amount conditional on giving
